In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import faiss
import time
from deepeval import evaluate
import json

/home/intern/MyWork/rag/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1531.19it/s]


In [3]:
with open("500DaysofSummer.txt", "r", encoding="utf-8") as file:
    text = file.read()

In [4]:
def chunk_text(text, chunk_size=800, overlap=300):
    chunks = []

    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks
chunks = chunk_text(text)

In [5]:
emb = model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

emb = emb.astype("float32")
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

In [13]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

In [14]:
from groq import Groq

client = Groq(api_key=api_key)

In [ ]:
import importlib, groq_deepeval
importlib.reload(groq_deepeval)
from groq_deepeval import GroqModel
groq_model = GroqModel(api_key, model="llama-3.1-8b-instant")


In [9]:
query = "What was douchebag referring to in the movie?"
query_embedding = model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding = query_embedding.reshape(1, -1).astype("float32")
retrieved_chunks = []
distances, indices = index.search(query_embedding, 10)
for idx in indices[0]:
    retrieved_chunks.append(chunks[idx])

context = "\n\n".join(retrieved_chunks)

distances, indices = index.search(query_embedding, 10)

In [ ]:
prompt = f"""
You are a question-answering assistant.

Use ONLY the provided context.

Rules:

1. Never use outside knowledge.
2. Never infer information that is not explicitly stated.
3. If the answer is missing, reply exactly:
   "No information available in the provided context."
4. After every answer, include the exact sentence(s) from the context that support your answer.
5. If no supporting sentence exists, return only:
   "No information available in the provided context."

Context:
{context}

Question:
{query}

"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print("Question:" ,query)
print("Answer:" ,response.choices[0].message.content)

In [85]:
with open("test_dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

In [6]:
def ask_rag(query):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, 10)
    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""
Use ONLY the context below.

If the answer is not present, say:
"No information available in the provided context."

Context:

{context}

Question:
{query}

Answer:
"""
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    return answer, retrieved_chunks

In [18]:
start = len(rag_output)

print("Resuming from question", start + 1)

Resuming from question 181


In [118]:
for item in dataset[start:]:

    question = item["question"]
    expected_answer = item["answer"]

    print(f"\nRunning: {question}")

    try:
        rag_answer, retrieved_chunks = ask_rag(question)

        rag_output.append({
            "question": question,
            "expected_answer": expected_answer,
            "rag_answer": rag_answer,
            "retrieved_chunks": retrieved_chunks
        })

        # Save immediately after each successful question
        with open("rag_outputs_partial.json", "w", encoding="utf-8") as f:
            json.dump(rag_output, f, indent=4, ensure_ascii=False)

        print("✓ Saved", len(rag_output), "answers")

    except Exception as e:
        print("Error:", e)
        print("Progress saved. Resume later.")
        break


Running: What is Tom's interaction like with Summer when they meet again at the bench?
✓ Saved 155 answers

Running: What is Summer's attitude towards Tom when they meet again?
✓ Saved 156 answers

Running: What is Tom's emotional state when he meets Summer again?
✓ Saved 157 answers

Running: What is the significance of the Angelus Plaza bench?
✓ Saved 158 answers

Running: What is Tom's career goal?
✓ Saved 159 answers

Running: What is the outcome of Tom's job search?
✓ Saved 160 answers

Running: Why did Summer dance with Tom at the wedding?
✓ Saved 161 answers

Running: What did Tom realize about his beliefs in destiny and true love?
✓ Saved 162 answers

Running: What book was Summer reading at the corner deli when she met her husband?
✓ Saved 163 answers

Running: What did Summer tell Tom about her marriage?
✓ Saved 164 answers

Running: What did Tom hope for Summer at the end of their conversation?
✓ Saved 165 answers

Running: Where did Tom go for a job interview?
✓ Saved 166 

In [16]:
import json
import os

# Load previous results
if os.path.exists("rag_outputs_partial.json"):
    with open("rag_outputs_partial.json", "r", encoding="utf-8") as f:
        rag_output = json.load(f)
else:
    rag_output = []

print("Already completed:", len(rag_output))

Already completed: 180


In [20]:
print(test_cases[0].__dict__)

IndexError: list index out of range

In [17]:
import json

from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    HallucinationMetric
)
from deepeval.test_case import LLMTestCase

faithfulness = FaithfulnessMetric(model=groq_model)

answer_relevancy = AnswerRelevancyMetric(model=groq_model)

hallucination = HallucinationMetric(model=groq_model)

In [18]:
import json
from deepeval.test_case import LLMTestCase

with open("rag_outputs_partial.json", "r", encoding="utf-8") as f:
    rag_outputs = json.load(f)

test_cases = []

for item in rag_outputs:

    question = item["question"]
    rag_answer = item["rag_answer"]
    expected_answer = item["expected_answer"]
    retrieved_chunks = item["retrieved_chunks"]

    tc = LLMTestCase(
        input=question,
        actual_output=rag_answer,
        expected_output=expected_answer,
        retrieval_context=retrieved_chunks,
        context=retrieved_chunks
    )

    test_cases.append(tc)

print(len(test_cases))

180


In [ ]:
small_test = test_cases[:20]

scores = {"faithfulness": [], "answer_relevancy": [], "hallucination": []}

for i, tc in enumerate(small_test):
    print(f"\n[{i+1}/{len(small_test)}] Q: {tc.input}")
    try:
        faithfulness.measure(tc)
        answer_relevancy.measure(tc)
        hallucination.measure(tc)

        scores['faithfulness'].append(faithfulness.score)
        scores['answer_relevancy'].append(answer_relevancy.score)
        scores['hallucination'].append(hallucination.score)

        print(f"  Faithfulness:     {faithfulness.score:.2f} ({'PASS' if faithfulness.is_successful() else 'FAIL'})")
        print(f"  Answer Relevancy: {answer_relevancy.score:.2f} ({'PASS' if answer_relevancy.is_successful() else 'FAIL'})")
        print(f"  Hallucination:    {hallucination.score:.2f} ({'PASS' if hallucination.is_successful() else 'FAIL'})")
    except Exception as e:
        print(f"  ERROR: {e}")

print("\n" + "="*40)
print("FINAL SUMMARY")
print("="*40)
time.sleep(10)
for metric, vals in scores.items():
    if vals:
        print(f"{metric:20s}: avg={sum(vals)/len(vals):.2f}, min={min(vals):.2f}, max={max(vals):.2f}")

/home/intern/MyWork/rag/.venv/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


[1/20] Q: What is the name of the boy in the story?


  Faithfulness:     0.83 (PASS)
  Answer Relevancy: 0.67 (PASS)
  Hallucination:    1.00 (FAIL)

[2/20] Q: Where is Tom from?
